In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 1: Install & Import\n",
    "%pip install langchain-community langchain-text-splitters langchain-chroma langchain-huggingface langchain-openai plotly scikit-learn gradio python-dotenv faiss-cpu\n",
    "print(\"✅ Setup complete\")\n",
    "\n",
    "import os, glob, time, numpy as np, plotly.graph_objects as go\n",
    "from plotly.subplots import make_subplots\n",
    "from pathlib import Path\n",
    "from dotenv import load_dotenv\n",
    "from sklearn.manifold import TSNE\n",
    "import gradio as gr\n",
    "import shutil, tempfile\n",
    "from collections import Counter\n",
    "\n",
    "# LangChain\n",
    "from langchain_community.document_loaders import DirectoryLoader, TextLoader\n",
    "from langchain_text_splitters import RecursiveCharacterTextSplitter\n",
    "from langchain_community.vectorstores import FAISS\n",
    "from langchain_huggingface import HuggingFaceEmbeddings\n",
    "from openai import OpenAI\n",
    "\n",
    "print(\"✅ Imports done\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 2: OpenRouter Setup\n",
    "load_dotenv()\n",
    "openrouter_api_key = os.getenv(\"OPENROUTER_API_KEY\") or os.environ.get(\"OPENROUTER_API_KEY\")\n",
    "openai_client = OpenAI(base_url=\"https://openrouter.ai/api/v1\", api_key=openrouter_api_key)\n",
    "MODEL = \"openai/gpt-4o-mini\"\n",
    "print(f\"✅ Using OpenRouter with {MODEL}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 3: Load/Create Knowledge Base\n",
    "KNOWLEDGE_BASE = \"knowledge-base\"\n",
    "if not os.path.isdir(KNOWLEDGE_BASE):\n",
    "    for candidate in [\"../../../week5/knowledge-base\", \"week5/knowledge-base\"]:\n",
    "        if os.path.isdir(candidate):\n",
    "            KNOWLEDGE_BASE = candidate\n",
    "            break\n",
    "\n",
    "# Create sample if none exists\n",
    "if not os.path.isdir(KNOWLEDGE_BASE):\n",
    "    os.makedirs(f\"{KNOWLEDGE_BASE}/employees\", exist_ok=True)\n",
    "    os.makedirs(f\"{KNOWLEDGE_BASE}/products\", exist_ok=True)\n",
    "    \n",
    "    with open(f\"{KNOWLEDGE_BASE}/employees/alex_lancaster.md\", 'w') as f:\n",
    "        f.write(\"# Alex Lancaster\\nCTO at Insurellm. Graduated from Manchester University.\")\n",
    "    with open(f\"{KNOWLEDGE_BASE}/products/carllm.md\", 'w') as f:\n",
    "        f.write(\"# CarLLM\\nAI auto insurance assistant.\")\n",
    "    with open(f\"{KNOWLEDGE_BASE}/products/homellm.md\", 'w') as f:\n",
    "        f.write(\"# HomeLLM\\nAI home insurance assistant.\")\n",
    "\n",
    "files = glob.glob(f\"{KNOWLEDGE_BASE}/**/*.md\", recursive=True)\n",
    "print(f\"📁 Found {len(files)} files\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 4: Load & Split Documents\n",
    "documents = []\n",
    "for folder in glob.glob(f\"{KNOWLEDGE_BASE}/*\"):\n",
    "    if os.path.isdir(folder):\n",
    "        loader = DirectoryLoader(folder, glob=\"*.md\", loader_cls=TextLoader)\n",
    "        docs = loader.load()\n",
    "        for doc in docs:\n",
    "            doc.metadata[\"type\"] = os.path.basename(folder)\n",
    "        documents.extend(docs)\n",
    "\n",
    "splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)\n",
    "chunks = splitter.split_documents(documents)\n",
    "print(f\"🔪 Created {len(chunks)} chunks\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 5: Create Embeddings with FAISS\n",
    "print(\"🔄 Creating embeddings...\")\n",
    "embeddings = HuggingFaceEmbeddings(model_name=\"all-MiniLM-L6-v2\")\n",
    "\n",
    "vectorstore = FAISS.from_documents(chunks, embeddings)\n",
    "print(f\"✅ Created FAISS vectorstore with {len(chunks)} chunks\")\n",
    "\n",
    "# Get vectors for visualization\n",
    "sample_size = min(50, len(chunks))\n",
    "sample_chunks = chunks[:sample_size]\n",
    "vectors = np.array([embeddings.embed_query(c.page_content) for c in sample_chunks])\n",
    "metadata = [{'type': c.metadata.get('type', 'unknown'), 'source': c.metadata.get('source', '')} for c in sample_chunks]\n",
    "print(f\"📊 Got {len(vectors)} vectors for viz\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 6: 2D Visualization\n",
    "if len(vectors) > 1:\n",
    "    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors)-1))\n",
    "    vectors_2d = tsne.fit_transform(vectors)\n",
    "    \n",
    "    fig = go.Figure()\n",
    "    colors = {'products':'blue', 'employees':'green', 'contracts':'red', 'company':'orange'}\n",
    "    for cat, color in colors.items():\n",
    "        mask = [m['type']==cat for m in metadata]\n",
    "        if any(mask):\n",
    "            fig.add_trace(go.Scatter(x=vectors_2d[mask,0], y=vectors_2d[mask,1], mode='markers', \n",
    "                                    name=cat, marker=dict(color=color)))\n",
    "    fig.update_layout(title=\"📊 Document Vectors\", width=800, height=500)\n",
    "    fig.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 7: Search Function\n",
    "def search(query, k=3):\n",
    "    docs = vectorstore.similarity_search_with_score(query, k=k)\n",
    "    print(f\"🔍 Query: '{query}'\")\n",
    "    for i, (doc, score) in enumerate(docs):\n",
    "        print(f\"{i+1}. Score: {score:.3f} - {doc.page_content[:50]}...\")\n",
    "    return docs"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 8: RAG Function\n",
    "def answer_question(message, history=None):\n",
    "    try:\n",
    "        docs = vectorstore.similarity_search(message, k=3)\n",
    "        context = \"\\n\\n\".join([d.page_content for d in docs])\n",
    "        \n",
    "        response = openai_client.chat.completions.create(\n",
    "            model=MODEL,\n",
    "            messages=[\n",
    "                {\"role\": \"system\", \"content\": f\"Answer using this context:\\n{context}\\nBe concise.\"},\n",
    "                {\"role\": \"user\", \"content\": message}\n",
    "            ]\n",
    "        )\n",
    "        return response.choices[0].message.content\n",
    "    except Exception as e:\n",
    "        return f\"Error: {str(e)}\"\n",
    "\n",
    "# Test\n",
    "print(\"\\n🤖 Testing RAG:\")\n",
    "print(answer_question(\"Who is Alex Lancaster?\", []))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 9: Gradio Chat Interface\n",
    "demo = gr.ChatInterface(\n",
    "    fn=answer_question,\n",
    "    title=\"🤖 Insurellm Assistant\",\n",
    "    description=\"Ask about employees, products, and contracts\"\n",
    ")\n",
    "\n",
    "print(\"\\n🚀 Launching chat...\")\n",
    "demo.launch(share=False, inbrowser=True)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 10: Performance Metrics\n",
    "questions = [\"Who is Alex Lancaster?\", \"What is CarLLM?\", \"Who is the CTO?\"]\n",
    "times, lengths = [], []\n",
    "\n",
    "for q in questions:\n",
    "    start = time.time()\n",
    "    ans = answer_question(q, [])\n",
    "    times.append(time.time() - start)\n",
    "    lengths.append(len(ans))\n",
    "    print(f\"✅ {q[:15]}... → {times[-1]:.2f}s\")\n",
    "\n",
    "if times:\n",
    "    fig = make_subplots(rows=1, cols=2)\n",
    "    fig.add_trace(go.Bar(x=[f\"Q{i+1}\" for i in range(len(questions))], y=times), row=1, col=1)\n",
    "    fig.add_trace(go.Bar(x=[f\"Q{i+1}\" for i in range(len(questions))], y=lengths), row=1, col=2)\n",
    "    fig.update_layout(title=\"📈 Performance\", height=400)\n",
    "    fig.show()\n",
    "    print(f\"Avg time: {np.mean(times):.2f}s\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.12.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}

In [ ]:
load_dotenv()
openrouter_api_key = os.getenv("OPENROUTER_API_KEY") or os.environ.get("OPENROUTER_API_KEY")
openai_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)
MODEL = "openai/gpt-4o-mini"
print(f"✅ Using OpenRouter with {MODEL}")

In [ ]:
# Find or create knowledge base
KNOWLEDGE_BASE = "knowledge-base"
if not os.path.isdir(KNOWLEDGE_BASE):
    for candidate in ["../../../week5/knowledge-base", "week5/knowledge-base"]:
        if os.path.isdir(candidate):
            KNOWLEDGE_BASE = candidate
            break

# Create sample if none exists
if not os.path.isdir(KNOWLEDGE_BASE):
    os.makedirs(f"{KNOWLEDGE_BASE}/employees", exist_ok=True)
    os.makedirs(f"{KNOWLEDGE_BASE}/products", exist_ok=True)
    
    with open(f"{KNOWLEDGE_BASE}/employees/alex_lancaster.md", 'w') as f:
        f.write("# Alex Lancaster\nCTO at Insurellm. Graduated from Manchester University.")
    with open(f"{KNOWLEDGE_BASE}/products/carllm.md", 'w') as f:
        f.write("# CarLLM\nAI auto insurance assistant.")
    with open(f"{KNOWLEDGE_BASE}/products/homellm.md", 'w') as f:
        f.write("# HomeLLM\nAI home insurance assistant.")

files = glob.glob(f"{KNOWLEDGE_BASE}/**/*.md", recursive=True)
print(f"📁 Found {len(files)} files")

In [ ]:
# Load documents
documents = []
for folder in glob.glob(f"{KNOWLEDGE_BASE}/*"):
    if os.path.isdir(folder):
        loader = DirectoryLoader(folder, glob="*.md", loader_cls=TextLoader)
        docs = loader.load()
        for doc in docs:
            doc.metadata["type"] = os.path.basename(folder)
        documents.extend(docs)

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"🔪 Created {len(chunks)} chunks")

In [ ]:
print("🔄 Creating embeddings...")
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Use FAISS instead of Chroma (avoids readonly issues)
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"✅ Created FAISS vectorstore with {len(chunks)} chunks")

# Get vectors for visualization (sample first 50 for speed)
sample_size = min(50, len(chunks))
sample_chunks = chunks[:sample_size]
vectors = np.array([embeddings.embed_query(c.page_content) for c in sample_chunks])
metadata = [{'type': c.metadata.get('type', 'unknown'), 'source': c.metadata.get('source', '')} for c in sample_chunks]
print(f"📊 Got {len(vectors)} vectors for visualization")

In [ ]:
if len(vectors) > 1:
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors)-1))
    vectors_2d = tsne.fit_transform(vectors)
    
    fig = go.Figure()
    colors = {'products':'blue', 'employees':'green', 'contracts':'red', 'company':'orange'}
    for cat, color in colors.items():
        mask = [m['type']==cat for m in metadata]
        if any(mask):
            fig.add_trace(go.Scatter(x=vectors_2d[mask,0], y=vectors_2d[mask,1], mode='markers', 
                                    name=cat, marker=dict(color=color)))
    fig.update_layout(title="📊 Document Vectors", width=800, height=500)
    fig.show()

In [ ]:
def search(query, k=3):
    docs = vectorstore.similarity_search_with_score(query, k=k)
    print(f"🔍 Query: '{query}'")
    for i, (doc, score) in enumerate(docs):
        print(f"{i+1}. Score: {score:.3f} - {doc.page_content[:50]}...")
    return docs

In [ ]:
def answer_question(message, history=None):
    try:
        docs = vectorstore.similarity_search(message, k=3)
        context = "\n\n".join([d.page_content for d in docs])
        
        response = openai_client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": f"Answer using this context:\n{context}\nBe concise."},
                {"role": "user", "content": message}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Test
print("\n🤖 Testing RAG:")
print(answer_question("Who is Alex Lancaster?", []))

In [ ]:
demo = gr.ChatInterface(
    fn=answer_question,
    title="🤖 Insurellm Assistant",
    description="Ask about employees, products, and contracts"
)

print("\n🚀 Launching chat...")
demo.launch(share=False, inbrowser=True)

In [ ]:
# Quick performance test
questions = ["Who is Alex Lancaster?", "What is CarLLM?", "Who is the CTO?"]
times, lengths = [], []

for q in questions:
    start = time.time()
    ans = answer_question(q, [])
    times.append(time.time() - start)
    lengths.append(len(ans))
    print(f"✅ {q[:15]}... → {times[-1]:.2f}s")

if times:
    fig = make_subplots(rows=1, cols=2)
    fig.add_trace(go.Bar(x=[f"Q{i+1}" for i in range(len(questions))], y=times), row=1, col=1)
    fig.add_trace(go.Bar(x=[f"Q{i+1}" for i in range(len(questions))], y=lengths), row=1, col=2)
    fig.update_layout(title="📈 Performance", height=400)
    fig.show()
    print(f"Avg time: {np.mean(times):.2f}s")